# Template 09a: Value-Adjusted Lift (all data)

Decile and 100-percentile value-adjusted lift charts for all data.

**Value-adjusted lift** (documentation/ModelMath_v2.md): the loss was divided by the depreciation
factor D upstream and the GLM carries D, so D is multiplied back in:
- Actual `A = (PP / GLM) x D`, Predicted `P = (PRED / GLM) x D`
- Rows sorted by P, exposure-weighted groups (equal exposure per group)
- Relativity = group mean / overall predicted mean (of that data); A/P = actual / predicted

`test_holdout` = test (05c) + holdout (08) stacked (disjoint folds). `train` is shown for comparison:
the model saw those losses, so train lift is optimistic.

Every chart PNG has a CSV with the same name next to it in `results/09/`.
CSV columns are described in documentation/ModelMath_v2.md, section "Stage 09 Lift Tables".

**Inputs:** results/05c_debug_{train,test}.parquet, scoring/holdout/predictions.parquet, aux file (Dep_factor)

**Outputs (results/09/):**
- 09a_summary.csv (rows, exposure, A/P, gini, top/bottom lift per dataset and bins)
- 09a_lift_{dataset}_{10,100}.png + .csv (chart and its decile / percentile table)

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import os
import sys
import yaml
import pandas as pd
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
project_root = setup_notebook_environment()
import lift_analysis as la

pd.set_option("display.width", 200)
print("########################################")
print("# STAGE 09a: VALUE-ADJUSTED LIFT")
print("########################################")

In [ ]:
pc_id = f"PC{open('current.pc').read().strip()}"
with open(f"{config_path}/config.yaml") as f:
    cfg = yaml.safe_load(f)
output_base = cfg["machines"][pc_id]["paths"]["output_path"]
model_name = cfg["experiment"]["name"]
vl = cfg.get("value_lift", {})
value_adjust = vl.get("value_adjust", True)
datasets = vl.get("datasets", ["test_holdout", "train"])
segment_file = vl.get("segment_file", "config/company_segment.csv")
min_group_exposure = vl.get("min_group_exposure", 5000)
out_dir = la.stage_dir(output_base)  # results/09: chart PNG + table CSV side by side
label = "value-adjusted" if value_adjust else "ratio (no Dep_factor)"
print(f"Model: {model_name} | output: {out_dir}")
print(f"Datasets: {datasets} | value_adjust: {value_adjust}")

In [ ]:
bins_list = vl.get("bins", [10, 100])
for f in os.listdir(out_dir):  # old 09a files (bins / datasets may change)
    if f.startswith("09a_"):
        os.remove(f"{out_dir}/{f}")
summaries, charts = [], {}
for ds in datasets:
    df = la.lift_frame(output_base, cfg, pc_id, ds, value_adjust=value_adjust)
    if df is None:
        print(f"[WARN] {ds}: no data, skipped")
        continue
    print(f"\n=== {ds}: {len(df):,} rows, exposure {df['weight'].sum():,.0f} ===")
    tables = {b: la.ntile_lift(df, b) for b in bins_list}
    for b, t in tables.items():
        name = "decile" if b == 10 else f"{b}-percentile" if b == 100 else f"{b}-group"
        charts[(ds, b)] = la.save_lift(t, f"{model_name} - {ds}: {label} {name} lift",
                                       f"{out_dir}/09a_lift_{ds}_{b}")
    s = la.overall_summary(df, tables)
    s.insert(0, "dataset", ds)
    summaries.append(s)
    del df

if not summaries:
    raise RuntimeError(f"No lift data for {datasets} - run 05c (train/test) and 08 (holdout) first")
summary = pd.concat(summaries, ignore_index=True)
summary.insert(1, "value_adjust", value_adjust)
summary.to_csv(f"{out_dir}/09a_summary.csv", index=False)
print(f"Saved to {out_dir}: 09a_summary.csv + {len(charts)} charts (PNG + CSV)")
summary.round(3)

## Lift charts

In [ ]:
for (ds, b), path in charts.items():
    display(Image(filename=path))
    if b == 10:
        t = pd.read_csv(path.replace(".png", ".csv"))
        print(t.round(3).to_string(index=False))

In [ ]:
print("\n########################################")
print("# STAGE 09a: COMPLETE")
print("########################################")